# Social-Oracle — a quantitative teardown 🔬
### Mention event study · abnormal-return CARs · random-day null · momentum control · the fade · clustering bootstrap · name jackknife · micro-cap capacity

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Pump--and--fade: Confirmed](https://img.shields.io/badge/Pump--and--fade-Confirmed-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We take the social-trading claim seriously, then ask whether a public mention is anything more than a **small, late, reversing attention bump** once you net out a random day, the momentum the name already had, and realistic micro-cap costs.

> ⚠️ **Not investment advice.** Real run on **1,468** r/WallStreetBets surges (CC-BY `youyanggu/yolostocks-data`); abnormal-return event study with a random-day null, a momentum control, a calendar-block clustering bootstrap and a name jackknife — references in [`docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition — so this notebook still reads even if you skim the maths. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # study root (social_oracle/ lives there)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from social_oracle import data, mentions, eventstudy, benchmark, backtest, robustness

# No live feed ships with this study: we run the *method* on a synthetic universe
# with a baked-in pump-and-fade. Swap the next line for data.load_feed('mentions.csv')
# + data.build_panel(...) to run it for real.
panel, feed = data.synthetic_panel(seed=0)
events, coverage = mentions.to_events(feed, panel)
print(f"{len(panel)} names, {len(feed)} mentions -> {len(events)} clean events")
print("coverage:", coverage)


## 1 · The claim, as testable hypotheses

H₁: E[CAR_{0→h} | mention] > 0 on **abnormal** returns (name − market), with clustering-robust t > 2, h ∈ {1,5,21}.
H₁′ (the sharp one): that excess survives a **momentum control** and realistic **micro-cap costs**.
H₀: forward abnormal return ≈ 0, or fully explained by prior momentum.

In [ ]:
es = eventstudy.event_study(panel, events, horizon=21, pre=5)
es['summary'].loc[[-5,-1,0,1,5,10,21]]

## 3–4 · The random-day null

`p_greater` = P(a random basket of the same size, drawn from every (name, day) in the universe, beats the mention basket). Small ⇒ the mention adds something.

In [ ]:
benchmark.conditional_vs_unconditional(panel, events, horizons=(1,5,21), n_iter=2000)

## 4 · The momentum control — does 'mentioned' beat 'already hot'?

The confound that makes this study hard: attention follows performance, so a mention rides an existing run. We pit mentions against hot-streak events (a name in its top-decile trailing return) on the same forward abnormal returns. **A mention that clears the random-day null but not this is a momentum sensor.**

In [ ]:
hot = mentions.hot_streak_events(panel)
benchmark.excess_vs_alternative(panel, events, hot, horizons=(1,5,21), n_iter=2000)

## 4 · The fade, clustering, and concentration

**(a) The fade** — mean abnormal CAR by horizon; a peak that reverses is the tell.

In [ ]:
robustness.fade_curve(panel, events)

**(b) Clustering** — mentions arrive in hype waves; a meme week is one bet, not thirty. A calendar-block bootstrap gives the honest CI on the excess.

In [ ]:
{k: round(v,4) for k,v in robustness.block_bootstrap_excess(panel, events, horizon=5, n_iter=2000).items()}

**(c) Concentration** — drop the most-mentioned names one at a time. If the excess collapses, you found a stock, not a skill.

In [ ]:
robustness.name_jackknife(panel, events, horizon=5, top=3)

## 5 · The verdict, with the numbers

Collate the decisive cells: random-day `p_greater`, the momentum `gap` and its p-value, the block-bootstrap `p_excess_le_0`, the fade-curve peak-vs-month, the jackknife swing. On a real feed these fill the README's beat-5 stamps. Expected shape: **Signal `WEAK`** (a real but small, run-up-contaminated bump) → **Tradability `MIRAGE`** once beat 6 charges costs.

## 6 · Could you trade it — costs and capacity

Enter at the next open (you saw the tweet when everyone did), hold a fixed window, charge a micro-cap spread twice. Then ask how much size the names can even absorb before your own order is the move.

In [ ]:
res = backtest.run(panel, events, hold_days=10)
print({k:(round(v,4) if isinstance(v,float) else v) for k,v in res.stats.items()})
print('\ncost sweep (half-spread bps -> mean net trade):')
display(backtest.cost_sweep(panel, events))
print('capacity at a nominal 50bp edge:')
backtest.capacity(panel, events, edge_bps=50.0)

**The data-mining check.** Hold period, lookback, cooldown — try enough knobs and one cell shines. Deflate the best Sharpe for the number of configs tried.

In [ ]:
import itertools
rows = []
for hold in (3,5,10,21):
    r = backtest.run(panel, events, hold_days=hold)
    rows.append({'hold': hold, 'sharpe': r.stats.get('sleeve_sharpe', float('nan')),
                 'n': r.stats['n_trades']})
scan = pd.DataFrame(rows).sort_values('sharpe', ascending=False)
best = scan.iloc[0]
dsr = robustness.deflated_sharpe(best.sharpe, n_trials=len(scan), n_obs=int(best.n))
print(f"best hold={int(best.hold)}d Sharpe={best.sharpe:.2f}; deflated over {len(scan)} configs: {dsr:.3f}")
scan

## 7 · Going further

- **Short the fade** — test the inverted trade directly, net of micro-cap borrow.
- **Beta-estimated abnormal return** to replace the β=1 market adjustment.
- **Conviction / first-mention / pile-on** splits of the feed.
- **A real, survivorship-clean feed** — the one input that turns this prior into a verdict.

Engine: [`../../../quantlab/`](../../../quantlab/). Method: [`METHODOLOGY.md`](../../../METHODOLOGY.md).